# Notebook 1.7: Second-Order Coherence and Photon Anti-Bunching
### Module 1: Single-Mode Quantum States of Light (Module Capstone)
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_1_states_of_light/09_second_order_coherence_g2.ipynb)

---

### Prerequisites
- Notebook 1.1: Fock States, Ladder Operators, and Truncation Artifacts
- Notebook 1.2: Coherent States, Displacement, and Poissonian Statistics
- Notebook 1.3: Squeezed Vacuum and Quadrature Noise Reduction
- Notebook 1.4: Displaced Squeezed States and Phase Space Interference
- Notebook 1.5: Thermal States and Mixed Radiation Fields
- Notebook 1.6: The Mandel Q-Parameter and Non-Classicality Criteria

### Pedagogical Learning Objectives
1. **The Second-Order Degree of Coherence:** Define $g^{(2)}(\tau)$ and zero-delay coherence $g^{(2)}(0) = \frac{\langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle}{\langle \hat{a}^\dagger \hat{a} \rangle^2} = 1 + \frac{Q}{\langle \hat{n} \rangle}$.
2. **The Classical Cauchy-Schwarz Bound:** Prove why classical electromagnetic wave intensity strictly obeys $g^{(2)}(0) \ge 1$ and $g^{(2)}(\tau) \le g^{(2)}(0)$, establishing **photon anti-bunching** ($g^{(2)}(0) < 1$) as a defining experimental hallmark of quantum light.
3. **The Hanbury Brown and Twiss (HBT) Setup:** Model the 50:50 beam splitter coincidence interferometer used in laboratories worldwide to measure single-photon purity.
4. **Time-Resolved Coherence Dynamics:** Simulate $g^{(2)}(\tau)$ for single quantum emitters ($1 - e^{-|\tau|/\tau_{\text{sp}}}$), coherent laser light ($g^{(2)}(\tau) \equiv 1$), and thermal chaotic light ($1 + e^{-2|\tau|/\tau_c}$, photon bunching).
5. **Experimental Imperfections & The $g^{(2)}(0) < 0.5$ Criterion:** Simulate detector timing jitter and dark count noise, deriving the universal experimental threshold for pure single-photon emission.
6. **Module 1 Grand Capstone Synthesis:** Compare all seven single-mode quantum states across their quantum numbers, noise geometry, Mandel $Q$, and $g^{(2)}(0)$.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
import scipy.signal as signal
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. Zero-Delay Second-Order Coherence $g^{(2)}(0)$

In quantum optics, the **second-order degree of temporal coherence** $g^{(2)}(\tau)$ characterizes intensity-intensity correlations and the conditional probability of detecting a photon at time $t+\tau$ given that a photon was detected at time $t$:

$$
g^{(2)}(\tau) = \frac{\langle \hat{a}^\dagger(t) \hat{a}^\dagger(t+\tau) \hat{a}(t+\tau) \hat{a}(t) \rangle}{\langle \hat{a}^\dagger(t) \hat{a}(t) \rangle \langle \hat{a}^\dagger(t+\tau) \hat{a}(t+\tau) \rangle}
$$

For a stationary single-mode field evaluated at zero time delay ($\tau = 0$):

$$
g^{(2)}(0) = \frac{\langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle}{\langle \hat{a}^\dagger \hat{a} \rangle^2} = \frac{\langle \hat{n}(\hat{n}-1) \rangle}{\langle \hat{n} \rangle^2}
$$

---

### Direct Identity with the Mandel $Q$-Parameter

Using the relation $\hat{n}(\hat{n}-1) = \hat{n}^2 - \hat{n}$:
$$
\langle \hat{n}(\hat{n}-1) \rangle = \langle \hat{n}^2 \rangle - \langle \hat{n} \rangle = \text{Var}(n) + \langle \hat{n} \rangle^2 - \langle \hat{n} \rangle
$$
Dividing by $\langle \hat{n} \rangle^2$:
$$
g^{(2)}(0) = 1 + \frac{\text{Var}(n) - \langle \hat{n} \rangle}{\langle \hat{n} \rangle^2} = 1 + \frac{Q}{\langle \hat{n} \rangle}
$$

This remarkable identity establishes the deep unity between photon counting variance ($Q$) and two-photon coincidence rates ($g^{(2)}(0)$):
- $g^{(2)}(0) < 1 \iff Q < 0$ (**Anti-bunched / Sub-Poissonian light**)
- $g^{(2)}(0) = 1 \iff Q = 0$ (**Coherent / Poissonian light**)
- $g^{(2)}(0) > 1 \iff Q > 0$ (**Bunched / Super-Poissonian light**)

---

### Proof: The Classical Cauchy-Schwarz Inequality

In classical electrodynamics, an optical field has intensity $I(t) \ge 0$. The classical second-order correlation is:
$$
g^{(2)}_{\text{classical}}(0) = \frac{\overline{I(t)^2}}{\overline{I(t)}^2}
$$
According to the Cauchy-Schwarz inequality for real positive random variables:
$$
\overline{I(t)^2} \cdot \overline{1^2} \ge \left( \overline{I(t) \cdot 1} \right)^2 \quad \Longrightarrow \quad \overline{I(t)^2} \ge \overline{I(t)}^2
$$
Dividing by $\overline{I(t)}^2$:
$$
g^{(2)}_{\text{classical}}(0) \ge 1
$$

Furthermore, the classical correlation theorem requires that the autocorrelation of a stationary process is maximized at zero delay:
$$
g^{(2)}_{\text{classical}}(\tau) \le g^{(2)}_{\text{classical}}(0)
$$

**The Quantum Violation:**
If an optical experiment measures:
$$
g^{(2)}(0) < 1 \quad \text{or} \quad g^{(2)}(\tau) > g^{(2)}(0)
$$
it is mathematically impossible to explain the result with classical electromagnetic wave theory!


In [ ]:
# Numerical Calculation of g^(2)(0) for Arbitrary Quantum States

def g2_zero_delay(state, N):
    """Computes zero-delay second-order coherence g^(2)(0) from state vector or density matrix."""
    n_diag = np.arange(N, dtype=np.float64)
    # Operator a^dag a = n
    n_op = np.diag(n_diag)
    # Operator (a^dag)^2 a^2 = n(n-1)
    n_n1_op = np.diag(n_diag * (n_diag - 1.0))
    
    if state.ndim == 1:
        psi = state / np.linalg.norm(state)
        mean_n = np.real(np.vdot(psi, n_op @ psi))
        mean_n_n1 = np.real(np.vdot(psi, n_n1_op @ psi))
    else:
        rho = state / np.real(np.trace(state))
        mean_n = np.real(np.trace(rho @ n_op))
        mean_n_n1 = np.real(np.trace(rho @ n_n1_op))
        
    if mean_n < 1e-12:
        return 0.0, mean_n
    
    g2 = mean_n_n1 / (mean_n**2)
    return g2, mean_n

# Benchmarking canonical quantum states
N_dim = 60

# 1. Single-Photon Fock state |1>
psi_1 = np.zeros(N_dim, dtype=np.complex128)
psi_1[1] = 1.0
g2_fock1, _ = g2_zero_delay(psi_1, N_dim)

# 2. Multi-photon Fock state |4>
psi_4 = np.zeros(N_dim, dtype=np.complex128)
psi_4[4] = 1.0
g2_fock4, _ = g2_zero_delay(psi_4, N_dim)

# 3. Coherent state |alpha> with alpha = 2.0
alpha_test = 2.0
p_coh = np.array([(alpha_test**(2*n)) * np.exp(-alpha_test**2) / math.factorial(n) for n in range(N_dim)])
psi_coh = np.sqrt(p_coh).astype(np.complex128)
g2_coh, _ = g2_zero_delay(psi_coh, N_dim)

# 4. Thermal state with n_bar = 2.0
n_bar_test = 2.0
p_th = (1.0 / (1.0 + n_bar_test)) * ((n_bar_test / (1.0 + n_bar_test)) ** np.arange(N_dim))
rho_th = np.diag(p_th).astype(np.complex128)
g2_th, _ = g2_zero_delay(rho_th, N_dim)

print("--- Zero-Delay Second-Order Coherence g^(2)(0) Verification ---")
print(f"Single-Photon State |1>: g^(2)(0) = {g2_fock1:.4f} (Expected: 0.0000, Perfect Anti-bunching)")
print(f"Fock State |4>:          g^(2)(0) = {g2_fock4:.4f} (Expected: 0.7500, Anti-bunched: 1 - 1/n)")
print(f"Coherent State |2.0>:    g^(2)(0) = {g2_coh:.4f} (Expected: 1.0000, Poissonian Limit)")
print(f"Thermal State (n=2):     g^(2)(0) = {g2_th:.4f} (Expected: 2.0000, HBT Photon Bunching)")

assert abs(g2_fock1 - 0.0) < 1e-8, "Fock |1> g2 test failed!"
assert abs(g2_fock4 - 0.75) < 1e-6, "Fock |4> g2 test failed!"
assert abs(g2_coh - 1.0) < 1e-4, "Coherent g2 test failed!"
assert abs(g2_th - 2.0) < 1e-4, "Thermal g2 test failed!"
print("All canonical g^(2)(0) values verified with high numerical precision!")


## 2. The Hanbury Brown and Twiss (HBT) Experiment

How is $g^{(2)}(\tau)$ measured experimentally in the laboratory?

In 1956, Robert Hanbury Brown and Richard Q. Twiss invented stellar intensity interferometry to measure the angular diameters of distant stars. In quantum optics, their interferometer was transformed into the gold-standard diagnostic for single-photon emitters:

```
                  Single-Photon Source
                          │
                          ▼
                    [50:50 Beam Splitter]
                         /     \
                        /       \
                       ▼         ▼
                 [Detector 1]  [Detector 2]
                       \         /
                        ▼       ▼
                     [Time-to-Digital Converter]
                               │
                               ▼
                   Coincidence Histogram G^(2)(τ)
```

---

### Physical Mechanism of Photon Anti-Bunching

Consider a single two-level atom or quantum dot:
1. The atom absorbs a pump photon, exciting it to the upper state $|e\rangle$.
2. It spontaneously decays to the ground state $|g\rangle$, emitting a **single photon** that travels toward the 50:50 beam splitter.
3. Because the photon is an indivisible quantum wave packet, it can either reflect to Detector 1 **OR** transmit to Detector 2, but **never both simultaneously**!
4. To emit a second photon, the atom must first be re-excited by the pump laser, which requires a finite time $\tau_{\text{pump}}$.
5. Therefore, the probability of detecting two photons at zero delay ($\tau = 0$) drops to **zero**:
   $$
   g^{(2)}(\tau) = 1 - e^{-|\tau|/\tau_{\text{sp}}}
   $$
   where $\tau_{\text{sp}} = 1/\Gamma$ is the spontaneous emission lifetime.

---

### Three Regimes of Time-Resolved Coherence $g^{(2)}(\tau)$:

1. **Anti-bunched Light ($g^{(2)}(0) < 1$):**
   $g^{(2)}(\tau) = 1 - e^{-|\tau|/\tau_{\text{sp}}}$ (Photons arrive spaced apart; a dip appears at $\tau = 0$).
2. **Coherent Laser Light ($g^{(2)}(\tau) \equiv 1$):**
   $g^{(2)}(\tau) = 1.0$ for all $\tau$ (Photons arrive independently and randomly).
3. **Thermal / Chaotic Light ($g^{(2)}(0) = 2$):**
   $g^{(2)}(\tau) = 1 + e^{-2|\tau|/\tau_c}$ (Photons arrive in correlated bunches; a peak appears at $\tau = 0$).


In [ ]:
# Simulating HBT Coincidence Histograms g^(2)(tau) for Three Light Regimes

tau_axis = np.linspace(-6.0, 6.0, 300)  # Time delay in nanoseconds (ns)
tau_sp = 1.2                            # Emitter lifetime (ns)
tau_c = 1.5                             # Thermal coherence time (ns)

# 1. Ideal Single-Photon Emitter (Anti-bunching)
g2_antibunched = 1.0 - np.exp(-np.abs(tau_axis) / tau_sp)

# 2. Coherent Laser Beam
g2_laser = np.ones_like(tau_axis)

# 3. Thermal Chaotic Light (Bunching)
g2_bunched = 1.0 + np.exp(-2.0 * np.abs(tau_axis) / tau_c)

# Plotting: Multi-panel comparative analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Theoretical Second-Order Coherence g^(2)(tau)
ax1.plot(tau_axis, g2_bunched, 'r-', lw=2.5, label=r'Thermal Light: $g^{(2)}(\tau) = 1 + e^{-2|\tau|/\tau_c}$')
ax1.plot(tau_axis, g2_laser, 'g--', lw=2.2, label=r'Coherent Laser: $g^{(2)}(\tau) = 1.0$')
ax1.plot(tau_axis, g2_antibunched, 'b-', lw=2.5, label=r'Single-Photon Emitter: $g^{(2)}(\tau) = 1 - e^{-|\tau|/\tau_{\mathrm{sp}}}$')

# Shaded regions
ax1.axhline(1.0, color='gray', ls=':', lw=1.2)
ax1.axhspan(0.0, 1.0, color='lightblue', alpha=0.2, label='Anti-bunching (Quantum: $g^{(2)} < 1$)')
ax1.axhspan(1.0, 2.2, color='mistyrose', alpha=0.2, label='Bunching (Super-Poissonian: $g^{(2)} > 1$)')

ax1.plot(0, 0, 'bo', markersize=7)
ax1.plot(0, 2, 'ro', markersize=7)

ax1.set_xlim(-6, 6)
ax1.set_ylim(-0.05, 2.25)
ax1.set_xlabel(r"Time Delay $\tau$ (ns)")
ax1.set_ylabel(r"Second-Order Coherence $g^{(2)}(\tau)$")
ax1.set_title("Time-Resolved Coherence: Anti-Bunching vs. Bunching")
ax1.legend(loc='upper right', framealpha=0.92, fontsize=9)
ax1.grid(True)

# Panel 2: Simulated Laboratory HBT Coincidence Counts
# Add realistic Poissonian sampling noise to simulate a multichannel TDC histogram
n_trials = 8000
counts_anti = np.random.poisson(g2_antibunched * n_trials)
counts_laser = np.random.poisson(g2_laser * n_trials)
counts_bunched = np.random.poisson(g2_bunched * n_trials)

ax2.step(tau_axis, counts_bunched, 'r-', alpha=0.7, lw=1.5, label='Thermal Bunching Peak')
ax2.step(tau_axis, counts_laser, 'g--', alpha=0.7, lw=1.5, label='Laser Flat Background')
ax2.step(tau_axis, counts_anti, 'b-', alpha=0.8, lw=1.8, label='Single-Photon Anti-bunching Dip')

ax2.set_xlim(-6, 6)
ax2.set_xlabel(r"Delay $\tau$ (ns)")
ax2.set_ylabel(r"Coincidence Counts $N_{\mathrm{coinc}}(\tau)$")
ax2.set_title("Simulated HBT Laboratory Coincidence Histogram")
ax2.legend(loc='upper right', framealpha=0.92, fontsize=9)
ax2.grid(True)

plt.tight_layout()
plt.show()


## 3. Real-World Laboratory Imperfections: The $g^{(2)}(0) < 0.5$ Criterion

In an ideal theoretical world, a single-photon emitter has $g^{(2)}(0) \equiv 0$. In real experimental physics, however, measured values are always strictly greater than zero! Why?

Two major laboratory imperfections affect every HBT measurement:

---

### 1. Finite Detector Timing Jitter
Single-Photon Avalanche Diodes (SPADs) and Superconducting Nanowire Single-Photon Detectors (SNSPDs) possess finite timing jitter (typically $\sigma_{\text{jitter}} \approx 30\text{ ps}$ to $500\text{ ps}$). The measured coincidence signal is the convolution of the physical $g^{(2)}(\tau)$ with the detector's Instrument Response Function (IRF):

$$
g^{(2)}_{\text{meas}}(\tau) = \int_{-\infty}^\infty g^{(2)}(\tau - t) \cdot \frac{1}{\sqrt{2\pi}\sigma_j} e^{-t^2 / (2\sigma_j^2)} \, dt
$$

Because the IRF averages values near $\tau = 0$ with values at $|\tau| > 0$, timing jitter **washes out and raises the anti-bunching dip**!

---

### 2. Ambient Stray Light and Detector Dark Counts
Even in a darkened cryostat, detector thermal dark counts and background laser scatter produce uncorrelated coincidence events:

$$
g^{(2)}_{\text{exp}}(0) \approx 1 - \rho^2 \left[ 1 - g^{(2)}_{\text{meas}}(0) \right]
$$

where $\rho = \frac{S}{S + B}$ is the signal-to-background ratio ($S = \text{signal rate}$, $B = \text{background rate}$).

---

### Why $g^{(2)}(0) < 0.5$ is the Accepted Threshold for Single-Photon Purity

Consider a two-photon Fock state $|2\rangle$:
$$
g^{(2)}_{|2\rangle}(0) = 1 - \frac{1}{2} = 0.5
$$
For an equal classical mixture or two independent single-photon emitters entering the beam splitter:
$$
g^{(2)}(0) = 0.5
$$

Therefore, in the quantum technology and quantum key distribution (QKD) community, the criterion:

$$
g^{(2)}(0) < 0.5
$$

is universally mandated to prove that an emitter is predominantly operating in the **single-photon regime** where multi-photon emissions are statistically suppressed!


In [ ]:
# Simulating Detector Jitter Convolution and Background Noise

tau_fine = np.linspace(-5.0, 5.0, 500)
dt = tau_fine[1] - tau_fine[0]
tau_sp = 0.8  # Emitter lifetime = 800 ps

# Ideal anti-bunching
g2_ideal = 1.0 - np.exp(-np.abs(tau_fine) / tau_sp)

# Gaussian IRF with different jitter values
sigma_jitters = [0.1, 0.3, 0.6]  # 100 ps, 300 ps, 600 ps jitter
colors = ['#1f77b4', '#ff7f0e', '#d62728']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Timing Jitter Convolution
ax1.plot(tau_fine, g2_ideal, 'k--', lw=2.2, label=r'Ideal Emitter ($g^{(2)}(0) = 0$)')

for sigma_j, col in zip(sigma_jitters, colors):
    irf = np.exp(-0.5 * (tau_fine / sigma_j)**2)
    irf /= np.sum(irf)  # Normalize area to 1
    # Convolution
    g2_conv = signal.fftconvolve(g2_ideal, irf, mode='same')
    g2_zero = g2_conv[len(tau_fine)//2]
    ax1.plot(tau_fine, g2_conv, color=col, lw=2.2,
             label=rf'Jitter $\sigma_j = {sigma_j*1000:.0f}\,$ps: $g^{{(2)}}(0) = {g2_zero:.3f}$')

ax1.axhline(0.5, color='purple', ls=':', lw=1.8, label=r'Quantum Single-Photon Threshold ($g^{(2)}=0.5$)')
ax1.set_xlim(-3, 3)
ax1.set_ylim(-0.02, 1.08)
ax1.set_xlabel(r"Delay $\tau$ (ns)")
ax1.set_ylabel(r"Measured $g^{(2)}(\tau)$")
ax1.set_title("Detector Jitter: Shallowing of the Anti-Bunching Dip")
ax1.legend(loc='lower right', framealpha=0.92, fontsize=9)
ax1.grid(True)

# Panel 2: Impact of Signal-to-Background Ratio (SBR)
sbr_values = [20.0, 8.0, 3.0, 1.5]  # Signal / Background
rho_values = [sbr / (sbr + 1.0) for sbr in sbr_values]

for sbr, rho, col in zip(sbr_values, rho_values, ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']):
    # g2_exp = 1 - rho^2 * (1 - g2_ideal)
    g2_noise = 1.0 - (rho**2) * (1.0 - g2_ideal)
    ax2.plot(tau_fine, g2_noise, color=col, lw=2.2,
             label=rf'SBR = {sbr:.1f} ($\rho={rho:.2f}$): $g^{{(2)}}(0) = {1 - rho**2:.3f}$')

ax2.axhline(0.5, color='purple', ls=':', lw=1.8, label=r'Single-Photon Threshold ($0.5$)')
ax2.set_xlim(-3, 3)
ax2.set_ylim(-0.02, 1.08)
ax2.set_xlabel(r"Delay $\tau$ (ns)")
ax2.set_ylabel(r"Measured $g^{(2)}(\tau)$")
ax2.set_title("Background Noise: Lifting the Zero-Delay Baseline")
ax2.legend(loc='lower right', framealpha=0.92, fontsize=9)
ax2.grid(True)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Multi-Photon Contamination Kills Quantum Advantage**
> In quantum key distribution (QKD) protocols like BB84, a malicious eavesdropper ("Eve") can exploit multi-photon pulses using the **photon-number splitting (PNS) attack**: Eve strips off one photon from a multi-photon pulse to measure it while allowing the remaining photon to pass undetected to Bob.
>
> Verifying $g^{(2)}(0) < 0.5$ certifies that two-photon emission events are strongly suppressed relative to Poissonian light, guaranteeing that the information encoded in single-photon pulses cannot be intercepted without introducing detectable bit errors!


In [ ]:
# Interactive Exploration: Laboratory HBT Single-Photon Simulator

def explore_hbt(lifetime_ns=1.0, jitter_ps=200.0, sbr=10.0):
    """Interactive simulation of measured HBT coincidence function."""
    tau = np.linspace(-4.0, 4.0, 400)
    sigma_j = jitter_ps / 1000.0  # Convert ps to ns
    
    # Ideal physical dip
    g2_phys = 1.0 - np.exp(-np.abs(tau) / lifetime_ns)
    
    # Convolution with Gaussian jitter IRF
    if sigma_j > 0.01:
        irf = np.exp(-0.5 * (tau / sigma_j)**2)
        irf /= np.sum(irf)
        g2_conv = signal.fftconvolve(g2_phys, irf, mode='same')
    else:
        g2_conv = g2_phys.copy()
        
    # Apply background dark count noise
    rho = sbr / (sbr + 1.0)
    g2_meas = 1.0 - (rho**2) * (1.0 - g2_conv)
    
    g2_zero_val = g2_meas[len(tau)//2]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Panel 1: Measured curve
    ax1.plot(tau, g2_meas, 'b-', lw=2.5, label=rf'Measured $g^{{(2)}}(\tau)$')
    ax1.plot(tau, g2_phys, 'k--', lw=1.5, alpha=0.6, label='Ideal Underlying Dip')
    ax1.axhline(0.5, color='red', ls=':', lw=2, label=r'Single-Photon Threshold ($0.5$)')
    ax1.plot(0, g2_zero_val, 'ro', markersize=7, label=rf'$g^{{(2)}}(0) = {g2_zero_val:.3f}$')
    
    ax1.set_xlim(-4, 4)
    ax1.set_ylim(-0.05, 1.1)
    ax1.set_xlabel(r"Delay $\tau$ (ns)")
    ax1.set_ylabel(r"$g^{(2)}(\tau)$")
    ax1.set_title(rf"HBT Coincidence: Lifetime={lifetime_ns:.1f}ns, Jitter={jitter_ps:.0f}ps, SBR={sbr:.1f}")
    ax1.legend(loc='lower right', fontsize=9)
    ax1.grid(True)
    
    # Panel 2: Gauge Meter
    status = "PURE SINGLE PHOTON" if g2_zero_val < 0.5 else "MULTI-PHOTON CONTAMINATED"
    color_bar = '#2ca02c' if g2_zero_val < 0.5 else '#d62728'
    
    ax2.barh([0], [g2_zero_val], color=color_bar, height=0.4, edgecolor='black')
    ax2.axvline(0.5, color='red', ls='--', lw=2, label='Threshold 0.5')
    ax2.set_xlim(0, 1.0)
    ax2.set_yticks([])
    ax2.set_xlabel(r"Measured $g^{(2)}(0)$")
    ax2.set_title(f"Quantum Purity Status: {status}")
    ax2.legend(loc='upper right')
    ax2.grid(True, axis='x')
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_hbt,
             lifetime_ns=FloatSlider(min=0.2, max=3.0, step=0.2, value=1.0, description='Lifetime (ns):'),
             jitter_ps=FloatSlider(min=20.0, max=600.0, step=20.0, value=200.0, description='Jitter (ps):'),
             sbr=FloatSlider(min=1.0, max=50.0, step=2.0, value=10.0, description='SBR:'))
else:
    # Static execution fallback for headless environments
    explore_hbt(lifetime_ns=1.0, jitter_ps=200.0, sbr=10.0)


## 4. Self-Check Exercises and Pedagogical Challenges

Test your understanding of second-order coherence and experimental single-photon verification with these analytical and computational challenges.

---

### Challenge 1: Second-Order Coherence of Squeezed Vacuum Light
For a squeezed vacuum state $|0, r\rangle$:
1. Using the normal ordering relation:
   $$
   \langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle = \sinh^2 r (2\cosh^2 r + \sinh^2 r) = 2\bar{n}^2 + \bar{n} + \bar{n}^2 = 3\bar{n}^2 + \bar{n}
   $$
   where $\bar{n} = \sinh^2 r$, prove that:
   $$
   g^{(2)}(0) = 3 + \frac{1}{\bar{n}} = 3 + \frac{1}{\sinh^2 r}
   $$
2. What is the value of $g^{(2)}(0)$ as $r \to 0$ ($\bar{n} \to 0$)? Explain physically why squeezed vacuum light exhibits **giant super-bunching** ($g^{(2)}(0) \gg 3$).

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
The zero-delay coherence is defined as:
$$
g^{(2)}(0) = \frac{\langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle}{\langle \hat{a}^\dagger \hat{a} \rangle^2}
$$
For squeezed vacuum, $\langle \hat{a}^\dagger \hat{a} \rangle = \sinh^2 r = \bar{n}$.
Evaluating the two-photon expectation value:
$$
\langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle = \text{Var}(n) + \bar{n}^2 - \bar{n}
$$
Recall that $\text{Var}(n) = 2\sinh^2 r \cosh^2 r = 2\bar{n}(\bar{n}+1) = 2\bar{n}^2 + 2\bar{n}$.
Therefore:
$$
\langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle = (2\bar{n}^2 + 2\bar{n}) + \bar{n}^2 - \bar{n} = 3\bar{n}^2 + \bar{n}
$$
Dividing by $\bar{n}^2$:
$$
g^{(2)}(0) = \frac{3\bar{n}^2 + \bar{n}}{\bar{n}^2} = 3 + \frac{1}{\bar{n}}
$$

**Part 2 Solution:**
As $r \to 0$, $\bar{n} \to 0$, which causes:
$$
\lim_{r \to 0} g^{(2)}(0) = \lim_{\bar{n} \to 0} \left( 3 + \frac{1}{\bar{n}} \right) = +\infty
$$
Physically, spontaneous parametric down-conversion generates photons **strictly in correlated pairs** ($|0\rangle, |2\rangle, |4\rangle \dots$). If a detector detects a photon at $t = 0$, it is guaranteed that a twin partner photon was born in the exact same event! Because the average photon rate $\bar{n}$ is near zero, the conditional probability of detecting a second photon immediately after the first is astronomically higher than random chance, resulting in **extreme photon super-bunching**!
</details>

---

### Challenge 2: Numerical Verification with QuTiP
Using QuTiP, verify that for a thermal state with $\bar{n} = 1.5$, $g^{(2)}(0) = 2.0000$, and for a displaced squeezed state with $\alpha = 3.0$ and $r = 0.5$, $g^{(2)}(0) < 1.0000$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
if HAS_QUTIP:
    import qutip as qt
    N = 60
    
    # 1. Thermal state
    rho_th = qt.thermal_dm(N, 1.5)
    a = qt.destroy(N)
    adag = a.dag()
    g2_th = qt.expect(adag * adag * a * a, rho_th) / (qt.expect(adag * a, rho_th)**2)
    print(f"QuTiP Thermal g^(2)(0): {g2_th:.4f}")
    assert abs(g2_th - 2.0) < 1e-4, "Thermal g2 test failed!"
    
    # 2. Amplitude squeezed state
    psi_sq = qt.displace(N, 3.0) * qt.squeeze(N, 0.5) * qt.fock(N, 0)
    g2_sq = qt.expect(adag * adag * a * a, psi_sq) / (qt.expect(adag * a, psi_sq)**2)
    print(f"QuTiP Squeezed g^(2)(0): {g2_sq:.4f}")
    assert g2_sq < 1.0, "State should be anti-bunched!"
    print("QuTiP validation confirmed!")
else:
    print("QuTiP not installed; run in Google Colab to execute this test.")
```
</details>


## 5. Module 1 Grand Capstone Summary and Syllabus Matrix

Congratulations! You have completed **Module 1: Single-Mode Quantum States of Light**.

---

### Master Comparison: The Landscape of Single-Mode Light States

| State | Notation | State Nature | $\Delta X_1 \Delta X_2$ | Photon Distribution $P_n$ | $\text{Var}(n)$ | Mandel $Q$ | $g^{(2)}(0)$ | Non-Classical? |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Fock State** | $|n\rangle$ | Pure | $\frac{2n+1}{4} \ge \frac{1}{4}$ | Deterministic: $\delta_{n, n_0}$ | $0$ | $-1$ | $1 - \frac{1}{n} < 1$ | **Yes** (Ultimate Quantum) |
| **Coherent State** | $|\alpha\rangle$ | Pure | $\frac{1}{4}$ (Min.) | Poissonian: $e^{-\bar{n}}\frac{\bar{n}^n}{n!}$ | $\bar{n}$ | $0$ | $1.0$ | **No** (Classical Benchmark) |
| **Squeezed Vacuum** | $|0, \xi\rangle$ | Pure | $\frac{1}{4}$ (Min.) | Even-Fock: $P_{2m+1}=0$ | $2\bar{n}(\bar{n}+1)$ | $1+2\bar{n} > 0$ | $3 + \frac{1}{\bar{n}} > 3$ | **Yes** (Quadrature Squeezed) |
| **Amp. Squeezed** | $|\alpha, r\rangle$ | Pure | $\frac{1}{4}$ (Min.) | Schleich-Wheeler fringes | $<\bar{n}$ | **$< 0$** | **$< 1$** | **Yes** (Sub-Poissonian) |
| **Phase Squeezed** | $|\alpha, re^{i\pi}\rangle$| Pure | $\frac{1}{4}$ (Min.) | Broadened distribution | $\gg \bar{n}$ | $> 0$ | $> 1$ | **Yes** (Phase Sensitive) |
| **Thermal State** | $\hat{\rho}_{\text{th}}$| Mixed | $\frac{2\bar{n}+1}{4} > \frac{1}{4}$| Bose-Einstein: $\frac{\bar{n}^n}{(1+\bar{n})^{n+1}}$ | $\bar{n} + \bar{n}^2$ | $+\bar{n} > 0$ | $2.0$ | **No** (Thermal Chaotic) |
| **Cat State** | $|\psi_+\rangle$ | Pure | $>\frac{1}{4}$ | Even-Fock interference | $>\bar{n}$ | $> 0$ | $> 1$ | **Yes** (Wigner Negative) |

---

### Key Takeaways of Module 1
1. **The Quantum State Spectrum:** We progressed from discrete particle eigenstates ($|n\rangle$), to classical wave-like coherent states ($|\alpha\rangle$), to noise-suppressed squeezed fields, to thermal ensembles ($\hat{\rho}_{\text{th}}$).
2. **Experimental Hallmarks of Non-Classicality:**
   - Quadrature squeezing: $\langle (\Delta X_1)^2 \rangle < 1/4$
   - Sub-Poissonian statistics: Mandel $Q < 0$ ($F < 1$)
   - Photon anti-bunching: $g^{(2)}(0) < 1$
3. **The HBT Milestone:** Single-photon emitters verified by $g^{(2)}(0) < 0.5$ form the cornerstone of quantum cryptography, linear optical computing, and quantum metrology.

---

### Primary References
- **Hanbury Brown, R., & Twiss, R. Q.** (1956). *Correlation between photons in two coherent beams of light*. Nature, 177, 27–29.
- **Kimble, H. J., Dagenais, M., & Mandel, L.** (1977). *Photon anti-bunching in resonance fluorescence*. Physical Review Letters, 39(11), 691–695.
- **Gerry, C. C., & Knight, P. L.** (2005). *Introductory Quantum Optics*. Cambridge University Press. Chapter 4: Non-classical Properties of Single-Mode Light.
- **Scully, M. O., & Zubairy, M. S.** (1997). *Quantum Optics*. Cambridge University Press. Chapter 4: Coherence and Correlation Functions.

---

### Forward Look: Module 2 — Phase Space and Quasiprobability Distributions
In **Module 2**, we generalize from 1D photon distributions to full 2D continuous quantum phase space:
- Notebook 2.1: The Wigner Quasiprobability Distribution $W(x, p)$
- Notebook 2.2: Wigner Negativity as a Metric for Quantum Non-Classicality
- Notebook 2.3: The Husimi Q-Function $Q(\alpha)$ and Heterodyne Measurement
- Notebook 2.4: The Glauber-Sudarshan P-Representation $P(\alpha)$
- Notebook 2.5: Schrödinger Cat States and Phase Space Interference
- Notebook 2.6: Quantum State Tomography and Wigner Function Reconstruction
